# Prompt Engineering et LangChain PromptTemplates

<figure>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7HnZLgyttvmbXmXf0tl_FQ/201033-AdobeStock-1254756887%20571x367.png" 
</figure>

## Objectifs

Après avoir terminé ce TP, vous serez en mesure de :

- **Comprendre les bases du prompt engineering** : Acquérir une base solide sur la manière de communiquer efficacement avec un LLM à l’aide de prompts, établissant ainsi les fondations pour des techniques plus avancées.

- **Maîtriser des techniques avancées de prompts** : Apprendre et appliquer des méthodes avancées de prompt engineering telles que le *few-shot* et l’apprentissage *self-consistent* afin d’optimiser les réponses du LLM.

- **Utiliser les modèles de prompts LangChain** : Devenir compétent dans l’utilisation des modèles de prompts de LangChain pour structurer et optimiser vos interactions avec les LLM.

- **Développer des agents LLM pratiques** : Acquérir les compétences nécessaires pour créer et implémenter des agents tels que des bots de questions-réponses et des outils de résumé de texte en utilisant les modèles de prompts LangChain, transformant la théorie en solutions pratiques.

Pour ce tp, vous devez générer une api key sur open ai et l'ajouter comme variable d'environnement ou remplacer le LLM par un LLM gratuit de Huggingface.


## 1. Librairies

In [3]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')



from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


In [4]:
# Import .env
import os
from dotenv import load_dotenv
load_dotenv()

True

## 2. Le LLM

In [21]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.5,
    top_p=0.2,
    #top_k=1,
    max_tokens=256,
    api_key=os.getenv("OPENAI_API_KEY")
)

## 3. Prompting

### 3.1 Prompt basique

Il s'agit de la forme la plus simple de prompt. Vous donnez un petit texte sans instruction ou formattage spécial et le LLM complètera votre texte.

In [19]:
prompt = "Paris is"
response = llm.invoke(prompt)

print(f"prompt: {prompt}\n")
print(f"response : {response.content}\n")

prompt: Paris is

response : Paris is the capital city of France. It is known for its rich history, iconic landmarks such as the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral, and its vibrant culture, art, fashion, and cuisine. Paris is often referred to as "The City of Light" and is one of the most popular tourist destinations in the world. If you want to know more specific information about Paris, feel free to ask!



In [22]:
prompt = "The future of artificial intelligence is"
response = llm.invoke(prompt)

print(f"prompt: {prompt}\n")
print(f"response : {response.content}\n")

prompt: The future of artificial intelligence is

response : The future of artificial intelligence (AI) is widely anticipated to be transformative across many aspects of society, technology, and the economy. Here are some key trends and possibilities often discussed:

1. **Increased Integration**: AI will become more deeply integrated into everyday life, from smart homes and personal assistants to healthcare, education, and transportation.

2. **Advancements in Machine Learning**: Continued improvements in algorithms, especially in areas like deep learning and reinforcement learning, will enable AI systems to perform more complex tasks with greater accuracy and efficiency.

3. **Ethical and Responsible AI**: As AI systems become more powerful, there will be a stronger emphasis on developing ethical guidelines, transparency, and fairness to prevent biases and ensure accountability.

4. **Automation and Workforce Impact**: AI-driven automation will transform industries by taking over rep

La réponse est tronquée à cause du paramètre max_tokens fixé à 256.

### 3.2 Zero shot prompt



[**Zero-shot prompting**](https://www.ibm.com/think/topics/zero-shot-prompting?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-In-Context+Learning+and+Prompt+Templates-v3-GenAIcourse_1741386184) est une technique où on demande au modèle d'exécuter une tâche via un prompt sans lui fournir d'exemple. Cette technique est utilise lorsque l'on veut tester la capacité du modèle à comprendre le contexte et les instructions sans être aidé.

- Une telle technique est aussi utile pour valider un modèle qu'on vient de fine tuner.

In [23]:
prompt = """Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

            Answer:
"""
response = llm.invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response.content}\n")

prompt: Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

            Answer:


response : False



In [24]:
prompt = """Translate this sentence in spanish:

'Paris Saint germain is the best club in the world.'

Answer:

"""
response = llm.invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response.content}\n")

prompt: Translate this sentence in spanish:

'Paris Saint germain is the best club in the world.'

Answer:



response : Paris Saint Germain es el mejor club del mundo.



### 3.3 One-shot prompting

[**One-shot prompting**](https://www.ibm.com/think/topics/one-shot-prompting?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-In-Context+Learning+and+Prompt+Templates-v3-GenAIcourse_1741386184), ici on introduit dans le prompt un exemple afin de guider le modèle dans ses réponses.

In [26]:
prompt = """Here is an example of translating a sentence from English to French:

            English: “How is the weather today?”
            French: “Comment est le temps aujourd'hui?”
            
            Now, translate the following sentence from English to French:
            
            English: “Where is the nearest supermarket?”
            
"""
response = llm.invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response.content}\n")

prompt: Here is an example of translating a sentence from English to French:

            English: “How is the weather today?”
            French: “Comment est le temps aujourd'hui?”
            
            Now, translate the following sentence from English to French:
            
            English: “Where is the nearest supermarket?”
            


response : French: « Où est le supermarché le plus proche ? »



In [27]:
prompt = """
Here is an example of extracting keywords from a sentence:

Sentence: "Cloud computing offers businesses flexibility, scalability, and cost-efficiency for their IT infrastructure needs."
Keywords: cloud computing, flexibility, scalability, cost-efficiency, IT infrastructure

---

Now, please extract the main keywords from the following sentence:

Sentence: "Sustainable agriculture practices focus on biodiversity, soil health, water conservation, and reducing chemical inputs."
Keywords:
"""

response = llm.invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response.content}\n")

prompt: 
Here is an example of extracting keywords from a sentence:

Sentence: "Cloud computing offers businesses flexibility, scalability, and cost-efficiency for their IT infrastructure needs."
Keywords: cloud computing, flexibility, scalability, cost-efficiency, IT infrastructure

---

Now, please extract the main keywords from the following sentence:

Sentence: "Sustainable agriculture practices focus on biodiversity, soil health, water conservation, and reducing chemical inputs."
Keywords:


response : Keywords: sustainable agriculture, biodiversity, soil health, water conservation, reducing chemical inputs



### 3.4 Few-shot prompt

[**Few-shot prompting**](https://www.ibm.com/think/topics/few-shot-prompting?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-In-Context+Learning+and+Prompt+Templates-v3-GenAIcourse_1741386184), ici on donne plus d'un exemple (en général entre 2 et 5) avant de demander au modèle de resoudre une tâche. Cette technique est pertinente pour des tâches plus compliquées.

In [30]:
prompt = """Here are few examples of classifying emotions in statements:

            Statement: 'I just won my first marathon!'
            Emotion: Joy
            
            Statement: 'I can't believe I lost my keys again.'
            Emotion: Frustration
            
            Statement: 'My best friend is moving to another country.'
            Emotion: Sadness
            
            Now, classify the emotion in the following statement:
            Statement: 'That movie was so scary I had to cover my eyes.’
            

"""
response = llm.invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response.content}\n")

prompt: Here are few examples of classifying emotions in statements:

            Statement: 'I just won my first marathon!'
            Emotion: Joy
            
            Statement: 'I can't believe I lost my keys again.'
            Emotion: Frustration
            
            Statement: 'My best friend is moving to another country.'
            Emotion: Sadness
            
            Now, classify the emotion in the following statement:
            Statement: 'That movie was so scary I had to cover my eyes.’
            



response : Emotion: Fear



### 3.5 Chain-of-thought (CoT) prompt

[**Chain-of-thought (CoT) prompting**](https://www.ibm.com/think/topics/chain-of-thoughts?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-In-Context+Learning+and+Prompt+Templates-v3-GenAIcourse_1741386184)encourage le modèle à décomposer des problèmes complexes en un raisonnement étape par étape avant d’arriver à une réponse finale. En montrant ou en demandant explicitement les étapes intermédiaires, cette technique améliore les capacités de résolution de problèmes du modèle et réduit les erreurs dans les tâches nécessitant un raisonnement en plusieurs étapes.

Le CoT est particulièrement efficace pour les problèmes mathématiques, le raisonnement logique et les tâches de prise de décision complexes.


In [31]:
prompt = """Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
            How many apples are there now?’

            Break down each step of your calculation

"""
response = llm.invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response.content}\n")

prompt: Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
            How many apples are there now?’

            Break down each step of your calculation



response : Let's break down the problem step-by-step:

**Problem:**  
A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. How many apples are there now?

---

### Step 1: Identify the initial quantity of apples
- The store **started with 22 apples**.

### Step 2: Account for apples sold
- The store **sold 15 apples**.
- Selling apples means the number of apples decreases.
- Calculate the remaining apples after selling:
  \[
  22 - 15 = 7
  \]
- So, after selling, the store has **7 apples** left.

### Step 3: Account for new delivery
- The store received a **new delivery of 8 apples**.
- This means the number of apples increases by 8.
- Add the new apples to the remaining apples:
  \[
  7 + 8 = 15
  \]

### Step 4: Final answer
- The store 

### 3.6 Self-consistency

[**Self-consistency**](https://www.promptingguide.ai/techniques/consistency) est une technique avancée dans laquelle le modèle génère plusieurs solutions ou réponses indépendantes au même problème, puis évalue ces différentes approches afin de déterminer le résultat le plus cohérent ou le plus fiable.

Cette méthode améliore la précision en exploitant la capacité du modèle à aborder les problèmes sous différents angles et à identifier la solution la plus robuste grâce à la comparaison et à la vérification

In [32]:
prompt = """When I was 6, my sister was half of my age. Now I am 70, what age is my sister?

            Provide three independent calculations and explanations, then determine the most consistent result.

"""
response = llm.invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response.content}\n")

prompt: When I was 6, my sister was half of my age. Now I am 70, what age is my sister?

            Provide three independent calculations and explanations, then determine the most consistent result.



response : Let's analyze the problem step-by-step with three independent methods and then determine the most consistent result.

---

### Problem Recap:
- When you were 6 years old, your sister was half your age.
- Now you are 70 years old.
- Question: How old is your sister now?

---

## Method 1: Direct Age Difference Calculation

1. When you were 6, your sister was half your age:
   \[
   \text{sister's age} = \frac{6}{2} = 3
   \]

2. The age difference between you and your sister is:
   \[
   6 - 3 = 3 \text{ years}
   \]

3. Since age difference remains constant, now that you are 70:
   \[
   \text{sister's age} = 70 - 3 = 67
   \]

**Result:** Sister is 67 years old.

---

## Method 2: Using Proportional Age Ratio

1. At age 6, sister's age was half, so ratio:
   \[
   \frac{\te

## 4. Application du prompting

Dans cette section, nous allons montrer comment exploiter les modèles de prompts de LangChain pour construire des applications pratiques avec des résultats cohérents et reproductibles. Chaque application suit un schéma commun en utilisant **l’approche LCEL** :

1. Définir le contenu ou le problème à traiter.

2. Créer un modèle (template) avec des variables pour le contenu dynamique.

3. Convertir le modèle en un **PromptTemplate** de LangChain.

4. Construire une chaîne en utilisant l’opérateur | pour connecter :

    - Les variables d’entrée

    - Le modèle de prompt

    - Le LLM

    - Un parseur de sortie

5. Invoquer la chaîne avec des entrées spécifiques pour générer les résultats.

Cette approche structurée vous permet de créer des composants réutilisables pour diverses tâches de NLP tout en conservant la flexibilité d’ajuster les paramètres et les entrées. Vous verrez comment ce modèle s’applique à différents cas d’usage.

### Introduction à LangChain

[LangChain](https://www.langchain.com/) est un framework puissant conçu pour simplifier le développement d’applications utilisant des modèles de langage. Conçu pour répondre aux défis liés à l’utilisation des LLM dans des contextes pratiques, LangChain fournit une interface standardisée permettant de connecter les modèles à diverses sources de données et environnements applicatifs.

LangChain agit comme une couche d’abstraction, facilitant la création d’applications LLM complexes sans avoir à gérer les détails bas-niveau de l’interaction avec les modèles. Ce framework est devenu un outil standard dans l’écosystème LLM, supportant un large éventail de cas d’usage, allant des chatbots aux systèmes d’analyse de documents.

Dans cette section, nous nous concentrerons sur les capacités des modèles de prompts de LangChain, en montrant comment elles peuvent être utilisées pour créer des interactions structurées et reproductibles avec les modèles de langage dans différents types d’applications.


### Prompt template

[Les modèles de prompts](https://python.langchain.com/v0.2/docs/concepts/#prompt-templates) sont un concept clé dans LangChain. Ils permettent de traduire les entrées utilisateur et les paramètres en instructions pour un modèle de langage. Ces modèles peuvent être utilisés pour guider la réponse d’un modèle, l’aidant à comprendre le contexte et à générer des sorties cohérentes et pertinentes basées sur le langage.

Un modèle de prompt agit comme une structure réutilisable pour générer des prompts avec des valeurs dynamiques. Il vous permet de définir un format cohérent tout en laissant des espaces réservés pour les variables qui changent selon chaque cas d’usage. Cette approche rend le processus de prompt plus systématique et plus facile à maintenir, surtout lorsque l’on travaille sur des applications complexes.

**La version moderne de LangChain (à partir de 2025) propose deux principales approches pour travailler avec les templates :**

- L’approche traditionnelle `LLMChain`  
- Le nouveau modèle LangChain Expression Language (LCEL) utilisant l’opérateur pipe `|` pour une composition plus flexible

LCEL est devenu le modèle recommandé pour créer des applications LangChain, car il offre une meilleure composabilité, une visualisation plus claire du flux de données et plus de flexibilité lors de la construction de chaînes complexes.

**Pour utiliser un modèle de prompt avec LCEL, vous suivez généralement ces étapes :**

- Définir votre template avec des variables entre accolades `{}`  
- Créer une instance de `PromptTemplate`  
- Construire une chaîne en utilisant l’opérateur pipe `|` pour connecter les composants  
- Invoquer la chaîne avec vos valeurs d’entrée



In [35]:
template = """Tell me a {adjective} joke about {content}.
"""
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['adjective', 'content'], input_types={}, partial_variables={}, template='Tell me a {adjective} joke about {content}.\n')

In [36]:
prompt.format(adjective="funny", content="AI")

'Tell me a funny joke about AI.\n'

In [37]:
from langchain_core.runnables import RunnableLambda


def format_prompt(variables):
    return prompt.format(**variables)

Le code suivant construit une chaîne en utilisant le modèle LCEL (LangChain Expression Language). Cette chaîne connecte les composants en utilisant l’opérateur pipe (`|`) pour créer un flux de traitement. La chaîne prend des variables d’entrée, les passe à travers le modèle de prompt, envoie le prompt formaté au LLM, et utilise un parseur de sortie de type chaîne pour retourner la réponse finale.


In [38]:
# Create the chain with explicit formatting
joke_chain = (
    RunnableLambda(format_prompt)
    | llm
    | StrOutputParser()
)

# Run the chain
response = joke_chain.invoke({"adjective": "funny", "content": "AI"})
print(response)

Sure! Here's a funny AI joke for you:

Why did the AI go to therapy?

Because it had too many neural issues!


In [39]:
response = joke_chain.invoke({"adjective": "sad", "content": "AI"})
print(response)

Why did the AI break up with its data?

Because it just couldn’t find the right *connection*—and now it’s left all alone, processing feelings it can’t understand.


#### Résumé

Construisons un agent dont le but est de faire le résumé de l'input.

In [40]:
content = """
    The rapid advancement of technology in the 21st century has transformed various industries, including healthcare, education, and transportation. 
    Innovations such as artificial intelligence, machine learning, and the Internet of Things have revolutionized how we approach everyday tasks and complex problems. 
    For instance, AI-powered diagnostic tools are improving the accuracy and speed of medical diagnoses, while smart transportation systems are making cities more efficient and reducing traffic congestion. 
    Moreover, online learning platforms are making education more accessible to people around the world, breaking down geographical and financial barriers. 
    These technological developments are not only enhancing productivity but also contributing to a more interconnected and informed society.
"""

template =  """Summarize the {content} in two sentences.
"""
prompt = PromptTemplate.from_template(template)

# LCEL chain
summarize_chain = (
    RunnableLambda(format_prompt)
    | llm
    | StrOutputParser()
)

# Run the chain
summary = summarize_chain.invoke({"content": content})
print(summary)

The rapid advancement of technologies like artificial intelligence, machine learning, and the Internet of Things has transformed industries such as healthcare, education, and transportation by improving efficiency, accessibility, and problem-solving capabilities. These innovations are enhancing productivity and fostering a more interconnected, informed global society.


In [ ]:
content = """
    The rapid advancement of technology in the 21st century has transformed various industries, including healthcare, education, and transportation. 
    Innovations such as artificial intelligence, machine learning, and the Internet of Things have revolutionized how we approach everyday tasks and complex problems. 
    For instance, AI-powered diagnostic tools are improving the accuracy and speed of medical diagnoses, while smart transportation systems are making cities more efficient and reducing traffic congestion. 
    Moreover, online learning platforms are making education more accessible to people around the world, breaking down geographical and financial barriers. 
    These technological developments are not only enhancing productivity but also contributing to a more interconnected and informed society.
"""

template = """Summarize the {content} in one sentence.
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
summarize_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
summary = summarize_chain.invoke({"content": content})
print(summary)

#### Q&A

Construisons un agent qui répond aux questions.

In [41]:
content = """
    The solar system consists of the Sun, eight planets, their moons, dwarf planets, and smaller objects like asteroids and comets. 
    The inner planets—Mercury, Venus, Earth, and Mars—are rocky and solid. 
    The outer planets—Jupiter, Saturn, Uranus, and Neptune—are much larger and gaseous.
"""

question = "Which planets in the solar system are rocky and solid?"

template = """
    Answer the {question} based on the {content}.
    Respond "Unsure about answer" if not sure about the answer.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
qa_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
answer = qa_chain.invoke({"question": question, "content": content})
print(answer)

Answer: The rocky and solid planets in the solar system are Mercury, Venus, Earth, and Mars.


#### Génération de code

Construisons un agent qui génère des requêtes SQL.

In [42]:
description = """
    Retrieve the names and email addresses of all customers from the 'customers' table who have made a purchase in the last 30 days. 
    The table 'purchases' contains a column 'purchase_date'
"""

template = """
    Generate an SQL query based on the {description}
    
    SQL Query:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
sql_generation_chain = (
    RunnableLambda(format_prompt) 
    | llm 
    | StrOutputParser()
)

# Run the chain
sql_query = sql_generation_chain.invoke({"description": description})
print(sql_query)

```sql
SELECT DISTINCT c.name, c.email
FROM customers c
JOIN purchases p ON c.customer_id = p.customer_id
WHERE p.purchase_date >= CURRENT_DATE - INTERVAL '30 days';
```


#### Service client

In [43]:
# Create the prompt template
template = """
Analyze the following product review:
"{review}"

Provide your analysis in the following format:
- Sentiment: (positive, negative, or neutral)
- Key Features Mentioned: (list the product features mentioned)
- Summary: (one-sentence summary)
"""

product_review_prompt = PromptTemplate.from_template(template)

# Create a formatting function
def format_review_prompt(variables):
    return product_review_prompt.format(**variables)

# Build the LCEL chain
review_analysis_chain = (
    RunnableLambda(format_review_prompt)
    | llm 
    | StrOutputParser()
)

# Process the reviews
reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]

for i, review in enumerate(reviews):
    print(f"==== Review #{i+1} ====")
    result = review_analysis_chain.invoke({"review": review})
    print(result)
    print()

==== Review #1 ====
- Sentiment: Positive  
- Key Features Mentioned: Camera quality, battery life, heating during gaming  
- Summary: The reviewer is very satisfied with the smartphone's camera and battery performance, though they note it tends to heat up slightly during gaming.

==== Review #2 ====
- Sentiment: negative  
- Key Features Mentioned: laptop speed, system stability (crashes), keyboard functionality, customer service  
- Summary: The laptop performs poorly with slow speed, frequent crashes, a malfunctioning keyboard after two months, and unhelpful customer service.

